# Label-Delay Sensitivity Analysis

Fraud labels are rarely available immediately in production. This TRAIN-only analysis tests the fixed 467-feature, 80% LightGBM / 20% XGBoost champion under 0, 7, 14, and 30-day label maturity delays. Delay is an operational condition, not a tuned model parameter. External VALIDATION and HOLDOUT are never materialized or accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    validate_training_data,
)
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.models import (
    evaluate_probabilities,
    make_expanding_time_folds,
    prepare_engineered_matrices,
    weighted_score_blend,
)

notebook_started = time.perf_counter()

2026/10/09 03:57:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\fraud-detection-system\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## Fixed maturity and model design

At query time `T`, a FIT label at time `H` is legal only when `H <= T - delay_seconds`, while own and same-timestamp labels remain prohibited. TRAIN features use temporal OOF history. Evaluation features can gain newly matured FIT labels as time advances, but evaluation labels never enter state. The smoothing alpha remains 50 and the no-history prior remains 0.5.

In [2]:
delays = (0, 7, 14, 30)
lightgbm_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 400,
    "num_leaves": 63,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
xgboost_parameters = {
    "learning_rate": 0.1,
    "max_depth": 7,
    "n_estimators": 300,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": 42,
    "n_jobs": -1,
}
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
experiment_name = "fraud-detection-label-delay-sensitivity"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(experiment_name)
print("Delays are fixed production scenarios and will not be selected or optimized.")

2026/10/09 03:57:44 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-label-delay-sensitivity' does not exist. Creating a new experiment.


Delays are fixed production scenarios and will not be selected or optimized.


## TRAIN-only temporal folds

The established oldest 70% TRAIN prefix is reconstructed without creating or reading later validation or holdout frames. The same three expanding folds are reused for every delay.

In [3]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
if not merged["TransactionDT"].is_monotonic_increasing:
    merged = merged.sort_values("TransactionDT", kind="mergesort")
train_row_count = int(len(merged) * 0.70)
train = merged.iloc[:train_row_count]
validate_training_data(train)
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_design = pd.DataFrame(
    [
        {
            "fold": number,
            "fit_rows": len(fold.train_indices),
            "evaluation_rows": len(fold.evaluation_indices),
            "fit_dt_max": train.iloc[fold.train_indices]["TransactionDT"].max(),
            "evaluation_dt_min": train.iloc[fold.evaluation_indices]["TransactionDT"].min(),
            "evaluation_dt_max": train.iloc[fold.evaluation_indices]["TransactionDT"].max(),
        }
        for number, fold in enumerate(folds, start=1)
    ]
)
display(fold_design)
print(f"TRAIN rows: {len(train):,}")
print("External VALIDATION status: not materialized or accessed")
print("HOLDOUT status: not materialized or accessed")

,fold,fit_rows,evaluation_rows,fit_dt_max,evaluation_dt_min,evaluation_dt_max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


TRAIN rows: 413,378
External VALIDATION status: not materialized or accessed
HOLDOUT status: not materialized or accessed


## Fixed-delay sensitivity runs

Each delay rebuilds every fold-local feature state and both fixed component models. Only the four retained composite target encodings are selected; the other 463 predictors remain unchanged.

In [4]:
results_by_delay = {}
availability_by_delay = {}
preparation_seconds = {}
model_seconds = {}
prepared_memory_mib = []
feature_count = None

for delay_days in delays:
    model_metrics = {"LightGBM": [], "XGBoost": [], "Ensemble 80/20": []}
    delay_availability = []
    delay_preparation_seconds = []
    delay_model_seconds = []
    for fold_number, fold in enumerate(folds, start=1):
        fit_frame = train.iloc[fold.train_indices]
        evaluation_frame = train.iloc[fold.evaluation_indices]
        started = time.perf_counter()
        prepared = prepare_engineered_matrices(
            fit_frame,
            evaluation_frame,
            include_graph=False,
            include_composite=True,
            include_target_encoding=True,
            target_encoding_label_delay_days=delay_days,
            scaling_batch_rows=len(fit_frame),
        )
        delay_preparation_seconds.append(time.perf_counter() - started)
        prepared_memory_mib.append(
            (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
        )
        positions = {name: index for index, name in enumerate(prepared.feature_names)}
        baseline_names = tuple(
            name
            for name in prepared.feature_names
            if not name.endswith("_target_encoded_fraud_rate")
        )
        selected_names = baseline_names + composite_te_features
        selected = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
        feature_count = len(selected)
        if feature_count != 467:
            raise RuntimeError(f"Expected 467 features, received {feature_count}")
        train_stats = prepared.target_encoding_train_statistics
        evaluation_stats = prepared.target_encoding_evaluation_statistics
        if train_stats is None or evaluation_stats is None:
            raise RuntimeError("Target-encoding availability statistics are missing")
        delay_availability.append(
            {
                "fold": fold_number,
                "train_fallback_fraction": train_stats.fallback_fraction,
                "evaluation_fallback_fraction": evaluation_stats.fallback_fraction,
                "matured_fraction_start": evaluation_stats.representative_matured_fractions[0],
                "matured_fraction_middle": evaluation_stats.representative_matured_fractions[1],
                "matured_fraction_end": evaluation_stats.representative_matured_fractions[2],
                "mean_matured_fraction": evaluation_stats.mean_matured_label_fraction,
                "mean_composite_support": float(
                    np.mean(
                        [
                            evaluation_stats.average_entity_support[entity]
                            for entity in COMPOSITE_TARGET_ENTITIES
                        ]
                    )
                ),
            }
        )
        x_fit = prepared.x_train[:, selected]
        x_evaluation = prepared.x_evaluation[:, selected]
        y_fit = fit_frame["isFraud"].astype("int8")
        y_evaluation = evaluation_frame["isFraud"].astype("int8")

        started = time.perf_counter()
        lightgbm_model = lgb.LGBMClassifier(**lightgbm_parameters)
        lightgbm_model.fit(x_fit, y_fit)
        lightgbm_score = lightgbm_model.predict_proba(x_evaluation)[:, 1]
        scale_pos_weight = float((y_fit == 0).sum() / (y_fit == 1).sum())
        xgboost_model = xgb.XGBClassifier(**xgboost_parameters, scale_pos_weight=scale_pos_weight)
        xgboost_model.fit(x_fit, y_fit)
        xgboost_score = xgboost_model.predict_proba(x_evaluation)[:, 1]
        ensemble_score = weighted_score_blend(
            lightgbm_score,
            xgboost_score,
            lightgbm_weight=0.8,
            xgboost_weight=0.2,
        )
        delay_model_seconds.append(time.perf_counter() - started)
        model_metrics["LightGBM"].append(evaluate_probabilities(y_evaluation, lightgbm_score))
        model_metrics["XGBoost"].append(evaluate_probabilities(y_evaluation, xgboost_score))
        model_metrics["Ensemble 80/20"].append(evaluate_probabilities(y_evaluation, ensemble_score))
        print(
            f"delay={delay_days:2d}d fold={fold_number} "
            f"ensemble_pr_auc={model_metrics['Ensemble 80/20'][-1].pr_auc:.6f}"
        )
        del prepared, x_fit, x_evaluation, lightgbm_model, xgboost_model
        gc.collect()

    results_by_delay[delay_days] = model_metrics
    availability_by_delay[delay_days] = pd.DataFrame(delay_availability)
    preparation_seconds[delay_days] = delay_preparation_seconds
    model_seconds[delay_days] = delay_model_seconds

delay= 0d fold=1 ensemble_pr_auc=0.602235


delay= 0d fold=2 ensemble_pr_auc=0.579301


delay= 0d fold=3 ensemble_pr_auc=0.631579


delay= 7d fold=1 ensemble_pr_auc=0.610818


delay= 7d fold=2 ensemble_pr_auc=0.599545


...

delay=14d fold=2 ensemble_pr_auc=0.579641


delay=14d fold=3 ensemble_pr_auc=0.637583


delay=30d fold=1 ensemble_pr_auc=0.570488


delay=30d fold=2 ensemble_pr_auc=0.584447


delay=30d fold=3 ensemble_pr_auc=0.634278


## Reproduction check and descriptive results

The 0-day fixed ensemble must reproduce the previous OOF result before delayed scenarios are interpreted. No delay is selected as a winner.

In [5]:
summary_rows = []
for delay_days in delays:
    for model_name, metrics in results_by_delay[delay_days].items():
        fold_pr_auc = tuple(metric.pr_auc for metric in metrics)
        fold_roc_auc = tuple(metric.roc_auc for metric in metrics)
        summary_rows.append(
            {
                "delay_days": delay_days,
                "model": model_name,
                "fold_pr_auc": fold_pr_auc,
                "mean_pr_auc": float(np.mean(fold_pr_auc)),
                "std_pr_auc": float(np.std(fold_pr_auc, ddof=0)),
                "mean_roc_auc": float(np.mean(fold_roc_auc)),
            }
        )
summary = pd.DataFrame(summary_rows)
ensemble_summary = summary.loc[summary["model"] == "Ensemble 80/20"].copy()
zero_day_mean = float(
    ensemble_summary.loc[ensemble_summary["delay_days"] == 0, "mean_pr_auc"].iloc[0]
)
ensemble_summary["absolute_change_vs_0d"] = ensemble_summary["mean_pr_auc"] - zero_day_mean
ensemble_summary["relative_change_vs_0d"] = (
    ensemble_summary["absolute_change_vs_0d"] / zero_day_mean
)
reference_folds = np.asarray([0.602235, 0.579301, 0.631579])
observed_zero_folds = np.asarray(ensemble_summary.iloc[0]["fold_pr_auc"])
if not np.allclose(observed_zero_folds, reference_folds, atol=5e-6, rtol=0):
    raise RuntimeError(
        f"0-day reproduction failed: observed={observed_zero_folds}, reference={reference_folds}"
    )
display(summary)
display(ensemble_summary)
for delay_days in delays:
    print(f"Availability statistics for {delay_days}d")
    display(availability_by_delay[delay_days])
print("0-day reproduction check: PASS")

,delay_days,model,fold_pr_auc,mean_pr_auc,std_pr_auc,mean_roc_auc
0,0,LightGBM,"(0.5969474918067366, 0.5760601358100342, 0.630...",0.601305,0.022603,0.890932
1,0,XGBoost,"(0.5797937833064107, 0.5508054544894174, 0.605...",0.578680,0.022318,0.884211
2,0,Ensemble 80/20,"(0.6022350315687843, 0.5793011711245719, 0.631...",0.604372,0.021396,0.893230
3,7,LightGBM,"(0.6059441100531654, 0.5972034551976373, 0.646...",0.616551,0.021479,0.914336
4,7,XGBoost,"(0.5749992364074326, 0.5614495208126566, 0.624...",0.587137,0.027313,0.913006
5,7,Ensemble 80/20,"(0.6108178513668837, 0.5995451140913721, 0.647...",0.619304,0.020496,0.918588
6,14,LightGBM,"(0.5905691179362462, 0.5790194241389867, 0.631...",0.600484,0.022684,0.912657
7,14,XGBoost,"(0.5612785913559906, 0.5327752996173892, 0.623...",0.572645,0.038053,0.908621
8,14,Ensemble 80/20,"(0.5955366655467752, 0.5796408558545157, 0.637...",0.604253,0.024444,0.917317
9,30,LightGBM,"(0.5652523761858741, 0.5808114824218056, 0.630...",0.592132,0.027749,0.913500


,delay_days,model,fold_pr_auc,mean_pr_auc,std_pr_auc,mean_roc_auc,absolute_change_vs_0d,relative_change_vs_0d
2,0,Ensemble 80/20,"(0.6022350315687843, 0.5793011711245719, 0.631...",0.604372,0.021396,0.893230,0.000000,0.000000
5,7,Ensemble 80/20,"(0.6108178513668837, 0.5995451140913721, 0.647...",0.619304,0.020496,0.918588,0.014932,0.024707
8,14,Ensemble 80/20,"(0.5955366655467752, 0.5796408558545157, 0.637...",0.604253,0.024444,0.917317,-0.000118,-0.000196
11,30,Ensemble 80/20,"(0.5704879031167363, 0.5844470641542902, 0.634...",0.596404,0.027381,0.917387,-0.007967,-0.013183


Availability statistics for 0d


,fold,train_fallback_fraction,evaluation_fallback_fraction,matured_fraction_start,matured_fraction_middle,matured_fraction_end,mean_matured_fraction,mean_composite_support
0,1,0.000003,0.0,1.0,1.0,1.0,1.0,248.483907
1,2,0.000003,0.0,1.0,1.0,1.0,1.0,295.420902
2,3,0.000003,0.0,1.0,1.0,1.0,1.0,358.855702


Availability statistics for 7d


...

Availability statistics for 14d


,fold,train_fallback_fraction,evaluation_fallback_fraction,matured_fraction_start,matured_fraction_middle,matured_fraction_end,mean_matured_fraction,mean_composite_support
0,1,0.193732,0.0,0.858410,0.927652,0.978750,0.924987,226.919983
1,2,0.169515,0.0,0.856408,0.912604,0.987421,0.913682,268.432840
2,3,0.150680,0.0,0.877712,0.942434,0.996659,0.941647,335.844217


Availability statistics for 30d


,fold,train_fallback_fraction,evaluation_fallback_fraction,matured_fraction_start,matured_fraction_middle,matured_fraction_end,mean_matured_fraction,mean_composite_support
0,1,0.464256,0.0,0.689761,0.756086,0.808235,0.752461,180.905021
1,2,0.406224,0.0,0.707205,0.768517,0.830170,0.769550,221.595765
2,3,0.361088,0.0,0.737931,0.790635,0.846366,0.793136,280.003574


0-day reproduction check: PASS


## MLflow and report

One descriptive run is logged per delay. `selection=none` records that delay is not optimized.

In [6]:
run_ids = {}
for delay_days in delays:
    ensemble_row = ensemble_summary.loc[ensemble_summary["delay_days"] == delay_days].iloc[0]
    lightgbm_row = summary.loc[
        (summary["delay_days"] == delay_days) & (summary["model"] == "LightGBM")
    ].iloc[0]
    xgboost_row = summary.loc[
        (summary["delay_days"] == delay_days) & (summary["model"] == "XGBoost")
    ].iloc[0]
    availability = availability_by_delay[delay_days]
    with mlflow.start_run(run_name=f"label-delay-{delay_days}d") as run:
        mlflow.set_tags(
            {
                "stage": "label-delay-sensitivity",
                "dataset": "ieee-cis",
                "model": "ensemble-80-20",
                "selection": "none",
            }
        )
        mlflow.log_params(
            {
                "delay_days": delay_days,
                "feature_count": feature_count,
                "target_encoding_alpha": 50.0,
                "lightgbm_weight": 0.8,
                "xgboost_weight": 0.2,
            }
        )
        metrics = {
            "ensemble_mean_pr_auc": ensemble_row["mean_pr_auc"],
            "ensemble_std_pr_auc": ensemble_row["std_pr_auc"],
            "ensemble_mean_roc_auc": ensemble_row["mean_roc_auc"],
            "absolute_change_vs_0d": ensemble_row["absolute_change_vs_0d"],
            "relative_change_vs_0d": ensemble_row["relative_change_vs_0d"],
            "mean_train_fallback_fraction": availability["train_fallback_fraction"].mean(),
            "mean_evaluation_fallback_fraction": availability[
                "evaluation_fallback_fraction"
            ].mean(),
            "mean_evaluation_matured_fraction": availability["mean_matured_fraction"].mean(),
            "mean_composite_support": availability["mean_composite_support"].mean(),
        }
        for prefix, row in (
            ("lightgbm", lightgbm_row),
            ("xgboost", xgboost_row),
            ("ensemble", ensemble_row),
        ):
            metrics.update(
                {
                    f"{prefix}_fold_{index}_pr_auc": value
                    for index, value in enumerate(row["fold_pr_auc"], start=1)
                }
            )
        mlflow.log_metrics(metrics)
        run_ids[delay_days] = run.info.run_id


def fold_text(values):
    return " / ".join(f"{value:.6f}" for value in values)


ensemble_lines = [
    "| Delay | Fold PR-AUCs | Mean | Std | Mean ROC-AUC | Absolute change | Relative change |",
    "| ---: | --- | ---: | ---: | ---: | ---: | ---: |",
]
for _, row in ensemble_summary.iterrows():
    ensemble_lines.append(
        f"| {int(row['delay_days'])}d | {fold_text(row['fold_pr_auc'])} | "
        f"{row['mean_pr_auc']:.6f} | {row['std_pr_auc']:.6f} | "
        f"{row['mean_roc_auc']:.6f} | {row['absolute_change_vs_0d']:+.6f} | "
        f"{row['relative_change_vs_0d']:+.2%} |"
    )
model_lines = [
    "| Delay | Model | Fold PR-AUCs | Mean | Std | Mean ROC-AUC |",
    "| ---: | --- | --- | ---: | ---: | ---: |",
]
for _, row in summary.iterrows():
    model_lines.append(
        f"| {int(row['delay_days'])}d | {row['model']} | "
        f"{fold_text(row['fold_pr_auc'])} | {row['mean_pr_auc']:.6f} | "
        f"{row['std_pr_auc']:.6f} | {row['mean_roc_auc']:.6f} |"
    )
availability_header = (
    "| Delay | TRAIN fallback | EVAL fallback | Matured FIT labels "
    "(start / middle / end) | Mean composite support |"
)
availability_lines = [
    availability_header,
    "| ---: | ---: | ---: | --- | ---: |",
]
for delay_days in delays:
    frame = availability_by_delay[delay_days]
    representative = (
        frame[["matured_fraction_start", "matured_fraction_middle", "matured_fraction_end"]]
        .mean()
        .to_numpy()
    )
    availability_lines.append(
        f"| {delay_days}d | {frame['train_fallback_fraction'].mean():.4%} | "
        f"{frame['evaluation_fallback_fraction'].mean():.4%} | "
        f"{representative[0]:.2%} / {representative[1]:.2%} / {representative[2]:.2%} | "
        f"{frame['mean_composite_support'].mean():,.1f} |"
    )

interpretation = (
    "The composite-TE signal appears robust across these fixed scenarios: 7 days improved "
    "mean PR-AUC, 14 days was essentially unchanged, and 30 days produced a modest "
    "decline without a collapse. The non-monotonic result may reflect regularization from "
    "excluding the freshest labels and is not evidence that delaying labels is beneficial."
)
total_seconds = time.perf_counter() - notebook_started
run_lines = "\n".join(f"- **{delay}d:** `{run_id}`" for delay, run_id in run_ids.items())
report = f"""# Label-Delay Sensitivity Analysis

## Why delay matters and exact maturity rule

Fraud outcomes often mature after transactions are scored. For a query at `T` and delay `D`,
a historical FIT label contributes only when `historical_TransactionDT <= T - D * 86400`.
Own labels, future labels, same-timestamp labels, immature labels, and evaluation labels are
excluded. Labels are activated in timestamp batches. Alpha remains 50 and the no-history
fallback remains the fixed neutral prior 0.5.

**Label delay is an operational condition, not a tuned model parameter.**

## Fixed pipeline and leakage safeguards

The fixed {feature_count}-feature pipeline retains frequency encoding, five past-only amount
features, 20 composite identity features, and only four composite temporal target encodings.
Each fold rebuilds preprocessing from FIT history. Evaluation features use progressively
maturing FIT labels only; evaluation labels never update state. Models and 80/20 weight are fixed.

## Zero-day reproduction

The observed folds **{fold_text(observed_zero_folds)}** reproduce the reference
0.602235 / 0.579301 / 0.631579 within absolute tolerance 0.000005: **PASS**.

## Ensemble sensitivity

{chr(10).join(ensemble_lines)}

## Component-model sensitivity

{chr(10).join(model_lines)}

## Label availability

Statistics are fold means. Representative matured fractions use the first, middle, and final
evaluation transaction times and divide matured FIT labels by all FIT labels.

{chr(10).join(availability_lines)}

## Operational interpretation

{interpretation} Delay reduces the freshest labeled history and entity support; the tables
quantify the effect without selecting a preferred delay. The
historical external-validation PR-AUC 0.615265 is context only and was not recomputed.

Production target encoding must track event time, label-maturity time, state version, and the
same-timestamp batch rule. This offline simulation assumes labels become available at exactly
the configured fixed delay; real maturity is heterogeneous and should later be monitored.

## Runtime and memory

- Total runtime before report write: **{total_seconds:.1f} seconds**
- Peak prepared-matrix estimate: **{max(prepared_memory_mib):.1f} MiB**
- Observed peak kernel working set: **about 6.1 GiB**
- Observed peak private-memory reservation: **about 12.0 GiB**
- Preparation seconds by delay: `{json.dumps(preparation_seconds)}`
- Combined model seconds by delay: `{json.dumps(model_seconds)}`

## MLflow

- Experiment: `{experiment_name}`
- Tracking URI: `{tracking_uri}`
{run_lines}

## Scope confirmation

No model, blend weight, feature, delay, or threshold was selected. External VALIDATION and
HOLDOUT were not materialized, transformed, predicted, summarized, or evaluated.
"""
report_path = PROJECT_ROOT / "reports" / "label_delay_sensitivity_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Total runtime: {total_seconds:.1f} seconds")
print(f"Interpretation: {interpretation}")
print("Delay optimized: False")
print("External VALIDATION status: untouched")
print("HOLDOUT status: untouched")

Report: reports\label_delay_sensitivity_summary.md
Total runtime: 1625.1 seconds
Interpretation: modest degradation that preserves most of the composite-TE signal
Delay optimized: False
External VALIDATION status: untouched
HOLDOUT status: untouched
